In [17]:
import pandas as pd

def interview_summary(filepath: str) -> pd.DataFrame:
    """
    """
    # Load the CSV file
    df  = pd.read_csv(filepath)

    def max_consecutive(series):
        """Returns the longest consecutive streak of True values in a boolean series"""
        max_streak = 0
        current_streak = 0
        for val in series:
            if val:
                current_streak += 1
                max_streak = max(max_streak, current_streak)
            else:
                current_streak = 0
        return max_streak

    # Per-person aggregation
    def summarize(g):
        total = len(g)
        n_speaking = (g["speaker"].str.strip().str.lower() == "speaking").sum()
        n_listening = (g["speaker"].str.strip().str.lower() == "listening").sum()
        duration = (g["timestamp"].max() - g["timestamp"].min())
        n_success = (g["success"] == 1).sum()
        n_not_success = (g["success"] != 1).sum()
        n_high_confidence = (g["confidence"] > 0.7).sum()
        n_low_confidence = (g["confidence"] <= 0.7).sum()
        depression = (g["depression"]).iloc[0]

        max_consecutive_fail = max_consecutive(g["success"] == 0)
        max_consecutive_low_conf = max_consecutive(g["confidence"] <= 0.7)

        return pd.Series({
            "depression": "yes" if depression == 1 else "no",
            "total_records": total,
            "pct_speaking": round(n_speaking / total * 100, 2),
            "pct_listening": round(n_listening / total * 100, 2),
            "interview_duration": duration,
            "n_success": n_success,
            "n_not_success": n_not_success,
            "pct_not_success": round(n_not_success / total * 100, 2),
            "n_high_confidence": n_high_confidence,
            "n_low_confidence": n_low_confidence,
            "pct_low_confidence": round(n_low_confidence / total * 100, 2),
            "max_consecutive_fail": max_consecutive_fail,
            "max_consecutive_low_confidence": max_consecutive_low_conf
        })

    return df.groupby("person_ID").apply(summarize).reset_index()

summary = interview_summary("/Users/pietrorebecchi/Documents/THIRD YEAR/SIXTH SEMESTER/BACHELOR PROJECT/copy/data/combined_gaze_labeled.csv")
print(summary)

     person_ID depression  total_records  pct_speaking  pct_listening  \
0          300         no          11581         46.91          53.09   
1          301         no          20306         77.98          22.02   
2          302         no          14073         57.89          42.11   
3          303         no          25448         73.16          26.84   
4          304         no          18045         63.10          36.90   
..         ...        ...            ...           ...            ...   
184        488         no          21503         75.74          24.26   
185        489         no          12916         55.96          44.04   
186        490         no          13623         58.25          41.75   
187        491         no          20887         71.14          28.86   
188        492         no          22447         80.87          19.13   

     interview_duration  n_success  n_not_success  pct_not_success  \
0             584.66700      11581              0    

In [18]:
summary.head(20)

,person_ID,depression,total_records,pct_speaking,pct_listening,interview_duration,n_success,n_not_success,pct_not_success,n_high_confidence,n_low_confidence,pct_low_confidence,max_consecutive_fail,max_consecutive_low_confidence
0,300,no,11581,46.91,53.09,584.6670,11581,0,0.00,11581,0,0.00,0,0
1,301,no,20306,77.98,22.02,774.3667,20306,0,0.00,20306,0,0.00,0,0
2,302,no,14073,57.89,42.11,676.9330,13755,318,2.26,13791,282,2.00,120,119
3,303,no,25448,73.16,26.84,934.0670,25445,3,0.01,25446,2,0.01,2,1
4,304,no,18045,63.10,36.90,719.9997,16776,1269,7.03,16933,1112,6.16,72,72
5,305,no,45539,85.94,14.06,1651.1333,45304,235,0.52,45321,218,0.48,180,180
6,306,no,22152,70.22,29.78,797.6337,22144,8,0.04,22145,7,0.03,3,3
7,307,no,32091,82.82,17.18,1175.0000,32048,43,0.13,32053,38,0.12,13,13
8,308,yes,19352,80.10,19.90,805.7003,18850,502,2.59,18853,499,2.58,120,120
9,309,yes,12246,56.25,43.75,624.0997,12146,100,0.82,12150,96,0.78,83,83


In [19]:
summary_depressed = summary[summary["depression"] == "yes"]
summary_depressed.head(20)

,person_ID,depression,total_records,pct_speaking,pct_listening,interview_duration,n_success,n_not_success,pct_not_success,n_high_confidence,n_low_confidence,pct_low_confidence,max_consecutive_fail,max_consecutive_low_confidence
8,308,yes,19352,80.10,19.90,805.70030,18850,502,2.59,18853,499,2.58,120,120
9,309,yes,12246,56.25,43.75,624.09970,12146,100,0.82,12150,96,0.78,83,83
11,311,yes,14308,59.16,40.84,717.96700,14287,21,0.15,14287,21,0.15,7,7
19,319,yes,13152,64.76,35.24,599.76700,12236,916,6.96,12390,762,5.79,84,105
20,320,yes,16119,60.06,39.94,805.96667,16000,119,0.74,16004,115,0.71,32,32
21,321,yes,15620,62.64,37.36,759.13370,15620,0,0.00,15620,0,0.00,0,0
25,325,yes,21587,75.99,24.01,820.33300,21587,0,0.00,21587,0,0.00,0,0
30,330,yes,15104,66.51,33.49,695.73300,15011,93,0.62,15021,83,0.55,29,29
32,332,yes,16707,72.77,27.23,746.00030,16707,0,0.00,16707,0,0.00,0,0
35,335,yes,19984,77.50,22.50,785.36700,19969,15,0.08,19970,14,0.07,8,8


In [20]:
average_total_records_depressed = summary_depressed["total_records"].mean()
print(average_total_records_depressed)
average_pct_speaking_depressed = summary_depressed["pct_speaking"].mean()
print(average_pct_speaking_depressed)
average_pct_listening_depressed = summary_depressed["pct_listening"].mean()
print(average_pct_listening_depressed)
average_interview_duration_depressed = summary_depressed["interview_duration"].mean()
print(average_interview_duration_depressed)
average_pct_not_success_depressed = summary_depressed["pct_not_success"].mean()
print(average_pct_not_success_depressed)
average_pct_low_confidence_depressed = summary_depressed["pct_low_confidence"].mean()
print(average_pct_low_confidence_depressed)
max_max_consecutive_fail = summary_depressed["max_consecutive_fail"].max()
print(max_max_consecutive_fail)
max_max_consecutive_low_confidence = summary_depressed["max_consecutive_low_confidence"].max()
print(max_max_consecutive_low_confidence)

23328.51785714286
74.91982142857141
25.080178571428572
947.4820028571428
1.5933928571428573
1.4394642857142859
336
343


In [21]:
summary_no_depressed = summary[summary["depression"] == "no"]
summary_no_depressed.head(20)

,person_ID,depression,total_records,pct_speaking,pct_listening,interview_duration,n_success,n_not_success,pct_not_success,n_high_confidence,n_low_confidence,pct_low_confidence,max_consecutive_fail,max_consecutive_low_confidence
0,300,no,11581,46.91,53.09,584.6670,11581,0,0.00,11581,0,0.00,0,0
1,301,no,20306,77.98,22.02,774.3667,20306,0,0.00,20306,0,0.00,0,0
2,302,no,14073,57.89,42.11,676.9330,13755,318,2.26,13791,282,2.00,120,119
3,303,no,25448,73.16,26.84,934.0670,25445,3,0.01,25446,2,0.01,2,1
4,304,no,18045,63.10,36.90,719.9997,16776,1269,7.03,16933,1112,6.16,72,72
5,305,no,45539,85.94,14.06,1651.1333,45304,235,0.52,45321,218,0.48,180,180
6,306,no,22152,70.22,29.78,797.6337,22144,8,0.04,22145,7,0.03,3,3
7,307,no,32091,82.82,17.18,1175.0000,32048,43,0.13,32053,38,0.12,13,13
10,310,no,16374,73.02,26.98,782.9000,16064,310,1.89,16107,267,1.63,51,51
12,312,no,16123,73.59,26.41,719.0663,16070,53,0.33,16073,50,0.31,19,18


In [22]:
average_total_records_no_depressed = summary_no_depressed["total_records"].mean()
print(average_total_records_no_depressed)
average_pct_speaking_no_depressed = summary_no_depressed["pct_speaking"].mean()
print(average_pct_speaking_no_depressed)
average_pct_listening_no_depressed = summary_no_depressed["pct_listening"].mean()
print(average_pct_listening_no_depressed)
average_interview_duration_no_depressed = summary_no_depressed["interview_duration"].mean()
print(average_interview_duration_no_depressed)
average_pct_not_success_no_depressed = summary_no_depressed["pct_not_success"].mean()
print(average_pct_not_success_no_depressed)
average_pct_low_confidence_no_depressed = summary_no_depressed["pct_low_confidence"].mean()
print(average_pct_low_confidence_no_depressed)
max_max_consecutive_fail = summary_no_depressed["max_consecutive_fail"].max()
print(max_max_consecutive_fail)
max_max_consecutive_low_confidence = summary_no_depressed["max_consecutive_low_confidence"].max()
print(max_max_consecutive_low_confidence)

22288.120300751878
73.80075187969925
26.19924812030075
882.2456490977445
1.1439097744360902
0.9888721804511279
480
480


In [23]:
summary_number_depressed = len(summary[summary["depression"] == "yes"])
print(summary_number_depressed)

56


In [24]:
summary_number_no_depressed = len(summary[summary["depression"] == "no"])
print(summary_number_no_depressed)

133


In [25]:
print(summary_number_depressed + summary_number_no_depressed)

189


In [26]:
summary_number_no_success = summary["n_not_success"].sum()
print(summary_number_no_success)

53912


In [27]:
summary_number_low_confidence = summary["n_low_confidence"].sum()
print(summary_number_low_confidence)

47237


In [28]:
def get_consecutive_streak_rows(df, person_id, metric="success"):
    """
    """
    person_df = df[df["person_ID"] == person_id].reset_index(drop=True)

    if metric == "success":
        mask = person_df["success"] == 0
    elif metric == "confidence":
        mask = person_df["confidence"] <= 0.7
    else:
        raise ValueError('metric must be "success" or "confidence"')

    # Find the longest consecutive streak and track start/end index
    max_streak   = 0
    curr_streak  = 0
    best_start   = 0
    curr_start   = 0

    for i, val in enumerate(mask):
        if val:
            if curr_streak == 0:
                curr_start = i
            curr_streak += 1
            if curr_streak > max_streak:
                max_streak = curr_streak
                best_start = curr_start
        else:
            curr_streak = 0

    if max_streak == 0:
        print(f'No failing rows found for person {person_id} with metric "{metric}"')
        return pd.DataFrame()

    best_end = best_start + max_streak
    print(f"Longest streak: {max_streak} rows (index {best_start} to {best_end - 1})")
    return person_df.iloc[best_start:best_end]

df = pd.read_csv("/Users/pietrorebecchi/Documents/THIRD YEAR/SIXTH SEMESTER/BACHELOR PROJECT/copy/data/combined_gaze_labeled.csv")

# Longest consecutive failed success for person XXX
streak_rows = get_consecutive_streak_rows(df, person_id=310, metric="success")
print(streak_rows)

# Longest consecutive low confidence for person XXX
# streak_rows = get_consecutive_streak_rows(df, person_id=302, metric="confidence")
# print(streak_rows)

Longest streak: 51 rows (index 4937 to 4987)
       speaker  frame  timestamp  confidence  success  x_0  y_0  z_0  x_1  \
4937  Speaking   8169    272.267    0.370838        0  0.0  0.0 -1.0  0.0   
4938  Speaking   8170    272.300    0.348777        0  0.0  0.0 -1.0  0.0   
4939  Speaking   8171    272.333    0.374473        0  0.0  0.0 -1.0  0.0   
4940  Speaking   8172    272.367    0.381572        0  0.0  0.0 -1.0  0.0   
4941  Speaking   8173    272.400    0.323495        0  0.0  0.0 -1.0  0.0   
4942  Speaking   8174    272.433    0.358849        0  0.0  0.0 -1.0  0.0   
4943  Speaking   8175    272.467    0.371519        0  0.0  0.0 -1.0  0.0   
4944  Speaking   8176    272.500    0.366068        0  0.0  0.0 -1.0  0.0   
4945  Speaking   8177    272.533    0.359927        0  0.0  0.0 -1.0  0.0   
4946  Speaking   8178    272.567    0.409323        0  0.0  0.0 -1.0  0.0   
4947  Speaking   8179    272.600    0.378833        0  0.0  0.0 -1.0  0.0   
4948  Speaking   8180    272.63